# 07 — Classification Data Prep

**Goal:** Pivot from NER to **multi-class sequence classification** — given a CTI report, predict its category (e.g., ransomware / phishing / apt-campaign / vulnerability-report / supply-chain).

Before training, we prep the data carefully. Classification-specific concerns:
- **Class balance** — if 70% of reports are one class, a trivial model scores 70% accuracy
- **Stratified splits** — validation must contain all classes proportionally
- **Label consistency** — a single report rarely fits one bucket cleanly

## What you'll learn

1. Designing a category schema
2. Building a synthetic CTI report dataset
3. Inspecting class balance
4. Stratified train/val split
5. Tokenizing for sequence classification (different from NER!)
6. What `labels` looks like for classification vs NER

## Step 1 — Category schema

Keep it small to start. Five categories is typical for a first pass:

- `ransomware` — encryption/extortion incidents
- `phishing` — credential harvesting, spear-phish
- `apt-campaign` — state-sponsored / targeted intrusions
- `vulnerability-report` — CVE disclosures, patch advisories
- `supply-chain` — third-party compromise

Single-label only for now. (Multi-label — a report tagged both `phishing` and `apt-campaign` — is a separate exercise.)

## Step 2 — Synthetic CTI report snippets

Each example is a short paragraph (stand-in for a full report) and its category label.

In [1]:
raw = [
    ("Conti ransomware encrypted servers at a US healthcare provider, demanding $5M in Bitcoin. Backups were destroyed.", "ransomware"),
    ("LockBit 3.0 affiliates deployed encryption payloads after initial access via a VPN brute force.", "ransomware"),
    ("Ryuk operators exfiltrated data from a manufacturing firm before triggering encryption. Double extortion observed.", "ransomware"),
    ("BlackCat/ALPHV ransomware hit an energy company, crippling OT environments for 72 hours.", "ransomware"),

    ("Spear-phishing campaign impersonating DHL delivered credential harvesting pages targeting logistics employees.", "phishing"),
    ("A phishing wave used fake Microsoft 365 login pages hosted on compromised WordPress sites.", "phishing"),
    ("Attackers sent invoice-themed emails with HTML smuggling attachments that dropped NetSupport RAT.", "phishing"),
    ("Business email compromise targeted finance teams with lookalike domains and wire-transfer fraud requests.", "phishing"),

    ("APT29 conducted long-running espionage against a European foreign ministry using custom loaders and legitimate cloud services.", "apt-campaign"),
    ("Lazarus Group targeted cryptocurrency exchanges with trojanized trading applications and social-engineering lures.", "apt-campaign"),
    ("Turla operators maintained persistent access to a diplomatic target using ComRAT and satellite-based C2.", "apt-campaign"),
    ("Sandworm disrupted Ukrainian power infrastructure with wiper malware during coordinated attacks.", "apt-campaign"),

    ("CVE-2023-23397 in Microsoft Outlook allows NTLM credential theft via crafted calendar invites. Patch available.", "vulnerability-report"),
    ("A critical remote code execution flaw in Atlassian Confluence, tracked as CVE-2023-22515, is under active exploitation.", "vulnerability-report"),
    ("CVE-2024-3400 in PAN-OS GlobalProtect is being weaponized in the wild. Vendor released emergency hotfix.", "vulnerability-report"),
    ("Apache disclosed CVE-2021-44228 (Log4Shell), a JNDI injection vulnerability affecting billions of systems.", "vulnerability-report"),

    ("SolarWinds Orion platform was backdoored via trojanized DLL, affecting 18,000 customers worldwide.", "supply-chain"),
    ("Malicious npm packages masquerading as legitimate utilities exfiltrated developer credentials.", "supply-chain"),
    ("3CX desktop application was compromised through upstream build-server breach, distributing malware to users.", "supply-chain"),
    ("A PyPI supply-chain attack replaced a popular package with a variant that stole AWS credentials.", "supply-chain"),
]

print(f"Total examples: {len(raw)}")

Total examples: 20


## Step 3 — Class balance check

In [2]:
from collections import Counter

counts = Counter(label for _, label in raw)
for lbl, n in counts.most_common():
    print(f"  {lbl:<24} {n}")

  ransomware               4
  phishing                 4
  apt-campaign             4
  vulnerability-report     4
  supply-chain             4


Perfectly balanced here (4 per class) because we constructed it that way. Real CTI data is usually imbalanced — ransomware dominates, supply-chain is rare. Common fixes:
- **Oversample** minority classes (duplicate or augment)
- **Class weights** in the loss (`CrossEntropyLoss(weight=...)`)
- **Focal loss** for severe imbalance

Start with the plain setup; add these only if the per-class F1 shows you need to.

## Step 4 — Label maps

In [3]:
categories = sorted({label for _, label in raw})
label2id = {c: i for i, c in enumerate(categories)}
id2label = {i: c for c, i in label2id.items()}
print("label2id:", label2id)

label2id: {'apt-campaign': 0, 'phishing': 1, 'ransomware': 2, 'supply-chain': 3, 'vulnerability-report': 4}


## Step 5 — Stratified split

Random splits on small datasets can produce validation sets missing entire classes. Stratify to preserve proportions.

In [4]:
from sklearn.model_selection import train_test_split

texts = [t for t, _ in raw]
labels = [label2id[l] for _, l in raw]

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.25, stratify=labels, random_state=42
)

print("Train size:", len(train_texts))
print("Val size:", len(val_texts))
print("Val class distribution:", Counter(val_labels))

Train size: 15
Val size: 5
Val class distribution: Counter({2: 1, 0: 1, 4: 1, 1: 1, 3: 1})


## Step 6 — Tokenize for classification

Unlike NER, classification input is a **single string per example**, and the label is a single integer. No subword alignment needed.

In [5]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def build(texts, labels):
    return Dataset.from_dict({"text": texts, "label": labels})

ds = DatasetDict({
    "train": build(train_texts, train_labels),
    "validation": build(val_texts, val_labels),
})

def tok(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized_ds = ds.map(tok, batched=True)
print(tokenized_ds)
print("\nFirst example keys:", list(tokenized_ds["train"][0].keys()))

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 15
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5
    })
})

First example keys: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


## Step 7 — The label tensor shape

For classification, `labels` is a **single integer per example** — compare with NER's per-token list.

In [6]:
example = tokenized_ds["train"][0]
print("text:         ", example["text"])
print("input_ids len:", len(example["input_ids"]))
print("label:        ", example["label"], "->", id2label[example["label"]])

text:          SolarWinds Orion platform was backdoored via trojanized DLL, affecting 18,000 customers worldwide.
input_ids len: 24
label:         3 -> supply-chain


## Save prepared data for the next notebook

We persist the dataset so notebook 8 can pick it up without recomputing.

In [7]:
import json, os

os.makedirs("./cls-data", exist_ok=True)

tokenized_ds.save_to_disk("./cls-data/tokenized")
with open("./cls-data/label_map.json", "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)

print("Saved to ./cls-data/")

Saving the dataset (0/1 shards):   0%|          | 0/15 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5 [00:00<?, ? examples/s]

Saved to ./cls-data/


## Your turn — exercises

1. Create an **imbalanced** version (e.g., 8 ransomware, 1 supply-chain). Compute class weights: `weight_i = total / (num_classes * count_i)`. You'll use these in notebook 8.
2. Add a 6th category of your choice (e.g., `insider-threat`, `ddos`). Rebalance and re-split.
3. Replace the short snippets with longer multi-sentence paragraphs. Re-check token lengths — does `max_length=256` still cover most examples?

## Next notebook

**`08_classification_finetuning.ipynb`** — load the saved data, fine-tune `BertForSequenceClassification`, evaluate, and run inference.